In [1]:

from bolero.tl.model.mc.train import mCBaseTrainer

import pyBigWig
import pandas as pd
import numpy as np
import pysam

from bolero import init

init(num_cpus=32, object_store_memory_gb=160, verbose=True)

Enabled torch cudnn
Disabled ray lineage reconstruction and task retries


2024-08-13 12:38:13,593	INFO worker.py:1744 -- Started a local Ray instance. View the dashboard at 127.0.0.1:8265 


In [2]:
data_dir = '/large_experiments/zhoulab/project/seqmodel/data/HBAtest_mC_multicelltype_hg38/ExampleALLCDataset'

# region_bed_path = f'/large_experiments/zhoulab/project/seqmodel/data/HBAtest_mC_multicelltype_hg38/bed/MajorTypeDMR.2500.bed'
region_bed_path = f'/large_experiments/zhoulab/zhoujt/project/seqmodel/240804_region_mc/train_bed/ASC.bed'

atac_bw_path = "/large_experiments/zhoulab/hanliu/wmb/Li2023Science/old_annot/bigwig/ASC.bw"
mc_allc_path = '/large_experiments/zhoulab/project/seqmodel/data/HBAtest_mC_multicelltype_hg38/allc/ASC_0.CGN-Merge.allc.tsv.gz'

In [13]:
bed = pd.read_csv(region_bed_path, sep='\t', header=None)
bed[2] - bed[1]

0          47
1         536
2         642
3          73
4         233
         ... 
199995     55
199996    123
199997     38
199998    177
199999    185
Length: 200000, dtype: int64

In [3]:
from bolero import hg38_splits

In [4]:
data_dir = '/large_experiments/zhoulab/project/seqmodel/data/HBAtest_mC_multicelltype_hg38/'
bed_dir = '/large_experiments/zhoulab/zhoujt/project/seqmodel/240804_region_mc/train_bed/'
split_id = 0

config = {
    "output_dir": "mc_seq_model",
    "region_bed_path": f"{bed_dir}ASC.bed",
    "chrom_split": hg38_splits[split_id],
    "dataset_path": f"{data_dir}ExampleALLCDataset",
    "prefix": 'allc',
    "genome": "hg38",
    "optimizer": "adamw",
    "scheduler": True,
    "scheduler_type": "step",
    "train_batches": 2000, #default 2000
    "val_batches": 300, # default 300
    "max_epochs": 30,
    "patience": 5,
    "wandb_job_type": "train",
    "wandb_project": "bolero_mc_advance",
    "wandb_group": None,
    "wandb_name": "mCtrack_ATACtrack_newbedfile",
    # changing parameters below:
    "savename": "mc_base",
    "output_channels": 2
}

config = mCBaseTrainer.make_config(**config)

trainer = mCBaseTrainer(config)


Create dataset with config: {'dataset_path': '/large_experiments/zhoulab/project/seqmodel/data/HBAtest_mC_multicelltype_hg38/ExampleALLCDataset', 'prefix': 'allc', 'genome': 'hg38', 'max_jitter': 128, 'dna_length': 1840, 'signal_length': 1000, 'min_cov': 50, 'max_cov': 100000, 'low_cov_ratio': 0.1, 'batch_size': 64, 'shuffle_files': False, 'read_parquet_kwargs': None}
File shuffle is disabled!!!
Loading genome DNA one-hot encoding...


In [5]:
trainer.dataset.train()

In [6]:
mcatac_dataset = trainer.dataset

In [7]:
loader = mcatac_dataset.get_dataloader(
    chroms=["chr1"], region_bed_path=region_bed_path, n_batches=300, as_torch=False
)

for batch in loader:
    pass

for k, v in batch.items():
    print(k, v.dtype, v.shape)

del loader

Get dataloader with train mode


Metadata Fetch Progress 0:   0%|          | 0/9 [00:00<?, ?it/s]

Parquet Files Sample 0:   0%|          | 0/2 [00:00<?, ?it/s]

2024-08-13 12:38:24,268	INFO streaming_executor.py:108 -- Starting execution of Dataset. Full logs are in /tmp/ray/session_2024-08-13_12-38-11_409162_4163047/logs/ray-data
2024-08-13 12:38:24,268	INFO streaming_executor.py:109 -- Execution plan of Dataset: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadParquet] -> ActorPoolMapOperator[Filter(region_length_filter)->Map(CompressedBytesToTensor)] -> ActorPoolMapOperator[FlatMap(GenerateRegions)] -> ActorPoolMapOperator[MapBatches(FilterRegions)] -> ActorPoolMapOperator[MapBatches(FetchRegionOneHot)] -> ActorPoolMapOperator[MapBatches(_mc_frac)->MapBatches(FetchRegionBigWigs)] -> TaskPoolMapOperator[MapBatches(_concat_bw_chunks)->MapBatches(CropLastAxisWithJitter)->MapBatches(ReverseComplement)->MapBatches(drop_columns)->MapBatches(_print)] -> LimitOperator[limit=19264]


Created remote one-hot object at ObjectRef(00ffffffffffffffffffffffffffffffffffffff0100000001e1f505)
Data loader kwargs {'prefetch_batches': 3, 'local_shuffle_buffer_size': 2000, 'drop_last': True, 'batch_size': 64}


...

- MapBatches(_concat_bw_chunks)->MapBatches(CropLastAxisWithJitter)->MapBatches(ReverseComplement)->MapBatches…

- limit=19264 8:   0%|          | 0/57 [00:00<?, ?it/s]

Running 0:   0%|          | 0/57 [00:00<?, ?it/s]

(MapBatches(_concat_bw_chunks)->MapBatches(CropLastAxisWithJitter)->MapBatches(ReverseComplement)->MapBatches(drop_columns)->MapBatches(_print) pid=4163550) {'dna_one_hot': (206, 4, 1840), 'allc_mc': (206, 3, 1000), 'allc_cov': (206, 3, 1000), 'allc_mc_frac': (206, 3, 1000), 'atac_bw_values': (206, 1, 1000)}
(MapBatches(_concat_bw_chunks)->MapBatches(CropLastAxisWithJitter)->MapBatches(ReverseComplement)->MapBatches(drop_columns)->MapBatches(_print) pid=4163550) {'dna_one_hot': (385, 4, 1840), 'allc_mc': (385, 3, 1000), 'allc_cov': (385, 3, 1000), 'allc_mc_frac': (385, 3, 1000), 'atac_bw_values': (385, 1, 1000)} [repeated 14x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication for more options.)
(MapBatches(_concat_bw_chunks)->MapBatches(CropLastAxisWithJitter)->MapBatches(ReverseComplement)->MapBatches(drop_columns)->MapBatches(_

dna_one_hot float32 (64, 4, 1840)
allc_mc float32 (64, 3, 1000)
allc_cov float32 (64, 3, 1000)
allc_mc_frac float32 (64, 3, 1000)
atac_bw_values float32 (64, 1, 1000)


In [9]:
loader = mcatac_dataset.get_dataloader(
    chroms=["chr1"], region_bed_path=region_bed_path, n_batches=300, as_torch=False
)

for batch in loader:
    pass

for k, v in batch.items():
    print(k, v.dtype, v.shape)

del loader

Get dataloader with train mode


Metadata Fetch Progress 0:   0%|          | 0/9 [00:00<?, ?it/s]

Parquet Files Sample 0:   0%|          | 0/2 [00:00<?, ?it/s]

2024-08-13 12:40:07,558	INFO streaming_executor.py:108 -- Starting execution of Dataset. Full logs are in /tmp/ray/session_2024-08-13_12-38-11_409162_4163047/logs/ray-data
2024-08-13 12:40:07,558	INFO streaming_executor.py:109 -- Execution plan of Dataset: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadParquet] -> ActorPoolMapOperator[Filter(region_length_filter)->Map(CompressedBytesToTensor)] -> ActorPoolMapOperator[FlatMap(GenerateRegions)] -> ActorPoolMapOperator[MapBatches(FilterRegions)] -> ActorPoolMapOperator[MapBatches(FetchRegionOneHot)] -> ActorPoolMapOperator[MapBatches(_mc_frac)->MapBatches(FetchRegionBigWigs)] -> TaskPoolMapOperator[MapBatches(_concat_bw_chunks)->MapBatches(CropLastAxisWithJitter)->MapBatches(ReverseComplement)->MapBatches(drop_columns)->MapBatches(_print)] -> LimitOperator[limit=19264]


Data loader kwargs {'prefetch_batches': 3, 'local_shuffle_buffer_size': 2000, 'drop_last': True, 'batch_size': 64}


...

- MapBatches(_mc_frac)->MapBatches(FetchRegionBigWigs) 6:   0%|          | 0/57 [00:00<?, ?it/s]

- MapBatches(_concat_bw_chunks)->MapBatches(CropLastAxisWithJitter)->MapBatches(ReverseComplement)->MapBatches…

- limit=19264 8:   0%|          | 0/57 [00:00<?, ?it/s]

Running 0:   0%|          | 0/57 [00:00<?, ?it/s]

dna_one_hot float32 (64, 4, 1840)
allc_mc float32 (64, 3, 1000)
allc_cov float32 (64, 3, 1000)
allc_mc_frac object (64,)
atac_bw_values object (64,)


In [7]:
row_id = 10

In [8]:
batch_value = batch['atac_bw_values'][row_id].ravel()

In [9]:
chrom, coords = batch['region'][row_id].split(':')
start, end = map(int, coords.split('-'))
jitter = batch['jitter'][row_id][0]
chrom, start, end, jitter

('chr1', 14682002, 14684342, 22)

In [10]:
_input_center = (start + end) // 2
_output_radius = 500
start = _input_center - _output_radius + jitter
end = start + 1000
chrom, start, end

('chr1', 14682694, 14683694)

In [11]:
with pyBigWig.open(atac_bw_path) as bw:
    data = np.nan_to_num(bw.values(chrom, start, end, numpy=True))

In [12]:
data

array([0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 2., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 1., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 2.,
       0., 0., 0., 0., 0.

In [13]:
with pysam.TabixFile(mc_allc_path) as allc:
    data = []
    for row in allc.fetch(chrom, start, end):
        data.append(row.strip().split('\t'))
region_allc_raw = pd.DataFrame(data)

[W::hts_idx_load3] The index file is older than the data file: /large_experiments/zhoulab/project/seqmodel/data/HBAtest_mC_multicelltype_hg38/allc/ASC_0.CGN-Merge.allc.tsv.gz.tbi


In [15]:
region_allc_raw

,0,1,2,3,4,5,6
0,chr1,14682709,+,CGT,19,19,1
1,chr1,14683054,+,CGA,22,25,1
2,chr1,14683082,+,CGC,18,20,1
3,chr1,14683095,+,CGT,6,18,1
4,chr1,14683193,+,CGA,1,29,1
5,chr1,14683212,+,CGG,9,32,1
6,chr1,14683238,+,CGC,9,31,1
7,chr1,14683269,+,CGT,7,31,1
8,chr1,14683290,+,CGA,4,25,1
9,chr1,14683324,+,CGC,3,4,1
